# v9 — Ablation: MLP + MiniLM pré-treinado (sem fine-tuning)

**Objetivo:** isolar contribuição do backbone (MiniLM) vs. contrastive fine-tuning (SetFit).

| Versão | Encoder | Fine-tune? |
|---|---|---|
| v7 | distiluse-v2 | ❌ |
| v9 (este) | MiniLM | ❌ |
| v8 | MiniLM | ✅ |

Todos os runs usam os **mesmos labels do v5** (qwen2.5-14b).

In [ ]:
# ── Cell 1: Mount Drive + Clone repo privado ──────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, sys

# Ajuste esses caminhos
REPO_URL   = 'https://<TOKEN>@github.com/FabioMMaia/multilingual-tad-thesis.git'
CLONE_DIR  = '/content/multilingual-tad-thesis'
DATA_DIR   = '/content/drive/MyDrive/Projeto ML/2025/AD/third_setup/adaptative-text-anomaly-detection/data'
V5_DIR     = '/content/drive/MyDrive/Projeto ML/2026/Master/Multilingual-Text-Anomaly-Detection/data/llm_results/v5'

if not os.path.exists(CLONE_DIR):
    os.system(f'git clone {REPO_URL} {CLONE_DIR}')
else:
    os.system(f'cd {CLONE_DIR} && git pull')

os.chdir(CLONE_DIR)
sys.path.insert(0, f'{CLONE_DIR}/src')
print('CWD:', os.getcwd())

In [ ]:
# ── Cell 2: Install requirements ──────────────────────────────────────────
!pip install -q -r requirements.txt

In [ ]:
# ── Cell 3: Skip helper ───────────────────────────────────────────────────
import glob, pandas as pd

RESULTS_DIR = 'data/llm_results/v9'

def _already_done(dataset, strategy, n, seed):
    pattern = f'{CLONE_DIR}/{RESULTS_DIR}/{strategy}/N_{n}/{dataset}.csv'
    for f in glob.glob(pattern):
        try:
            df = pd.read_csv(f)
            match = df[
                (df['n_llm_calls'] == int(n)) &
                (df['seed']        == int(seed))
            ]
            if not match.empty:
                return True
        except Exception:
            pass
    return False

In [ ]:
# ── Cell 4: v9 Sweep (48 runs) ────────────────────────────────────────────
import subprocess, re, time, os

REENCODER  = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'

datasets   = ['tweets_hs', 'hatebr', '20_newsgroups', 'wikinews']
strategies = ['random', 'diversity']
ns         = ['50', '200']
seeds      = ['0', '1', '42']

total = len(datasets) * len(strategies) * len(ns) * len(seeds)  # 48
run   = 0

for seed in seeds:
    for dataset in datasets:
        for strategy in strategies:
            for n in ns:
                run += 1

                if _already_done(dataset, strategy, n, seed):
                    print(f'[{run:03d}/{total}] ↷ {dataset:<20} {strategy:<12} N={n:<4} seed={seed} (skip)', flush=True)
                    continue

                labels_csv = f'{V5_DIR}/{strategy}/N_{n}/{dataset}_llm_labels.csv'
                if not os.path.exists(labels_csv):
                    print(f'[{run:03d}/{total}] ✗ labels not found: {labels_csv}', flush=True)
                    continue

                cmd = [
                    'python', '-u',
                    'scripts/run_llm_active_loop.py',
                    '--project_path',       CLONE_DIR,
                    '--data_dir',           DATA_DIR,
                    '--dataset',            dataset,
                    '--strategy',           strategy,
                    '--n_llm_calls',        n,
                    '--seed',               seed,
                    '--device',             'cuda',
                    '--load_labels_from',   labels_csv,
                    '--load_labels_model',  'qwen2.5-14b',
                    '--no_setfit',
                    '--reencoder',          REENCODER,
                    '--semisup_model',      'mlp',
                    '--results_dir',        RESULTS_DIR,
                ]

                t0 = time.time()
                result = subprocess.run(cmd, capture_output=True, text=True)
                elapsed = time.time() - t0

                roc = re.search(r'ROC-AUC\s+\(test\)\s*:\s*([\d.]+)', result.stdout)
                roc_str = roc.group(1) if roc else 'N/A'
                status  = '✓' if result.returncode == 0 else '✗'

                print(f'[{run:03d}/{total}] {status} {dataset:<20} {strategy:<12} N={n:<4} seed={seed}  '
                      f'ROC={roc_str}  {elapsed/60:.1f}min', flush=True)

                if result.returncode != 0:
                    print(f'  STDERR: {result.stderr[-400:]}', flush=True)

print('\nDone.')

In [ ]:
# ── Cell 5: Análise v7 vs v9 vs v8 ───────────────────────────────────────
import pandas as pd, glob, numpy as np

def load_v(version):
    files = [f for f in glob.glob(f'{CLONE_DIR}/data/llm_results/{version}/**/*.csv', recursive=True)
             if 'llm_labels' not in f]
    dfs = []
    for f in files:
        df = pd.read_csv(f)
        df['version'] = version
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

df7 = load_v('v7')  # MLP + distiluse
df8 = load_v('v8')  # MLP + MiniLM fine-tuned
df9 = load_v('v9')  # MLP + MiniLM pretrained  ← este experimento

key = ['dataset', 'strategy', 'n_llm_calls', 'seed']

print('=== Média global por versão ===')
for name, df in [('v7 (distiluse)', df7), ('v9 (MiniLM, sem FT)', df9), ('v8 (MiniLM+SetFit)', df8)]:
    if not df.empty:
        print(f'  {name}: {df["roc_auc"].mean():.4f} ±{df["roc_auc"].std():.4f}')

# Comparação pareada por dataset
if not df7.empty and not df9.empty:
    m79 = df7[key + ['roc_auc']].merge(df9[key + ['roc_auc']], on=key, suffixes=('_v7', '_v9'))
    m79['backbone_gain'] = m79['roc_auc_v9'] - m79['roc_auc_v7']  # ganho do backbone
    print('\n=== Ganho do backbone MiniLM (v9 - v7) por dataset ===')
    print(m79.groupby('dataset')['backbone_gain'].agg(['mean','std']).round(4).to_string())
    print(f'Global: {m79["backbone_gain"].mean():.4f} ±{m79["backbone_gain"].std():.4f}')

if not df9.empty and not df8.empty:
    m98 = df9[key + ['roc_auc']].merge(df8[key + ['roc_auc']], on=key, suffixes=('_v9', '_v8'))
    m98['finetune_gain'] = m98['roc_auc_v8'] - m98['roc_auc_v9']  # ganho do fine-tuning
    print('\n=== Ganho do fine-tuning SetFit (v8 - v9) por dataset ===')
    print(m98.groupby('dataset')['finetune_gain'].agg(['mean','std']).round(4).to_string())
    print(f'Global: {m98["finetune_gain"].mean():.4f} ±{m98["finetune_gain"].std():.4f}')